# DSN AI Bootcamp 2026 Machine Learning Qualification Hackathon

## DSN Mart Product-Store Sales Prediction

This notebook documents my modelling work for the DSN AI Bootcamp 2026 Machine Learning Qualification Hackathon. The task is to predict `total_sales` for product-store observations in the competition test set.

The competition evaluates predictions using **Root Mean Squared Error (RMSE)**, where lower values indicate smaller prediction error.

I kept the workflow focused on the information available in the competition data: data quality, train/test structure, a small amount of useful exploratory analysis, leakage-free validation, model comparison, final training, and submission checks. The Kaggle test target was not available during model development, so the model was selected using only cross-validation on the training data.

## 1. Setup and configuration

I used a fixed random seed and five shuffled folds so that the experiments could be reproduced and compared under the same validation procedure.

The notebook accepts the competition files either from the project folder or from a `data/` subfolder. The competition CSV files are read without modifying their original contents.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor

RANDOM_STATE = 42
N_FOLDS = 5
TARGET = "total_sales"
ID_COL = "id"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

candidate_dirs = [Path("."), Path("data")]
DATA_DIR = next(
    (
        d for d in candidate_dirs
        if (d / "train.csv").exists()
        and (d / "test.csv").exists()
        and (d / "sample_submission.csv").exists()
    ),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find train.csv, test.csv and sample_submission.csv "
        "in the project folder or data/."
    )

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())
print("Random state:", RANDOM_STATE)
print("Number of folds:", N_FOLDS)

## 2. Load the competition files

The training data contain the observed `total_sales` target. The test data contain the product-store observations for which predictions are required. The sample submission shows the required output structure.

I first checked the dimensions and column structure before making any modelling decisions.

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nSubmission columns:")
print(sample_submission.columns.tolist())

display(train.head())
display(test.head())
display(sample_submission.head())

## 3. Basic data audit

I checked data types, missing values, duplicate records and identifier uniqueness. This was mainly to identify preprocessing requirements and make sure the competition files were structurally sound before modelling.

The audit also gives the first description of the target variable.

In [ ]:
audit = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(2),
    "n_unique": train.nunique(dropna=False)
}).sort_values(["missing", "n_unique"], ascending=[False, True])

display(audit)

print("Duplicate train rows:", train.duplicated().sum())
print("Duplicate test rows:", test.duplicated().sum())
print("Duplicate train IDs:", train[ID_COL].duplicated().sum())
print("Duplicate test IDs:", test[ID_COL].duplicated().sum())

print("\nTarget summary:")
display(train[TARGET].describe().to_frame().T)

## 4. Train and test structure

I compared the product and store identifiers across the two datasets because this affects how much the model can rely on previously observed entities.

I also checked exact product-store pair overlap. This is particularly relevant here because the prediction task contains combinations of products and stores rather than repeated observations of the same row.

In [ ]:
train_products = set(train["product_code"].dropna().unique())
test_products = set(test["product_code"].dropna().unique())
train_stores = set(train["store_code"].dropna().unique())
test_stores = set(test["store_code"].dropna().unique())

train_pairs = set(zip(train["product_code"], train["store_code"]))
test_pairs = set(zip(test["product_code"], test["store_code"]))

print("Unique products in train:", len(train_products))
print("Unique products in test:", len(test_products))
print("Products appearing in both:", len(train_products & test_products))
print("Unseen test products:", len(test_products - train_products))

print("\nUnique stores in train:", len(train_stores))
print("Unique stores in test:", len(test_stores))
print("Stores appearing in both:", len(train_stores & test_stores))

print("\nExact product-store pairs in train:", len(train_pairs))
print("Exact product-store pairs in test:", len(test_pairs))
print("Overlapping product-store pairs:", len(train_pairs & test_pairs))

numeric_candidates = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years"
]

distribution_comparison = pd.DataFrame(index=numeric_candidates)

for col in numeric_candidates:
    distribution_comparison.loc[col, "train_mean"] = train[col].mean()
    distribution_comparison.loc[col, "test_mean"] = test[col].mean()
    distribution_comparison.loc[col, "train_median"] = train[col].median()
    distribution_comparison.loc[col, "test_median"] = test[col].median()

print("\nTrain/test numeric distribution comparison:")
display(distribution_comparison.round(3))

## 5. Focused exploratory analysis

I kept the exploratory analysis tied to modelling decisions rather than producing a large collection of plots.

The main questions were:

- How is `total_sales` distributed?
- How different are sales across stores?
- Do product categories show different sales levels?
- Is product price associated with sales?
- Which variables contain substantial missingness?

These checks help determine whether the model needs to capture strong store effects, nonlinear relationships, categorical effects, and missing values.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(train[TARGET], bins=40)
ax.set_title("Distribution of Total Sales")
ax.set_xlabel("Total Sales")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Target skewness:", round(train[TARGET].skew(), 3))

store_summary = (
    train.groupby(["store_code", "store_format"], dropna=False)[TARGET]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

print("\nSales by store:")
display(store_summary.round(2))

category_summary = (
    train.groupby("product_category", dropna=False)[TARGET]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)

print("\nSales by product category:")
display(category_summary.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(train["product_price"], train[TARGET], alpha=0.35)
ax.set_title("Product Price vs Total Sales")
ax.set_xlabel("Product Price")
ax.set_ylabel("Total Sales")
plt.tight_layout()
plt.show()

price_sales_corr = train[["product_price", TARGET]].corr().iloc[0, 1]
print("Pearson correlation between product price and total sales:",
      round(price_sales_corr, 3))

missing_summary = train.isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing_summary["missing_pct"] = (
    missing_summary["missing_count"] / len(train) * 100
).round(2)

print("\nVariables with missing values:")
display(missing_summary[missing_summary["missing_count"] > 0])

### EDA interpretation

The target is positively skewed, so a small number of observations have substantially higher sales than the centre of the distribution.

Store-level differences are pronounced, which makes the store variables important predictors. Product price also has a noticeable positive linear association with sales, although the scatter shows that price alone does not explain the target.

Two variables have substantial missingness: `product_weight_kg` and `store_size`. Rather than dropping those rows, I retained them and allowed the modelling pipelines to handle missing values. The categorical model also receives explicit missing-category values where required.

These observations support using a model that can represent nonlinear effects and categorical differences without requiring a very large one-hot encoded feature space.

## 6. Validation strategy

The true target values for `test.csv` are hidden by Kaggle, so I needed an internal estimate of generalisation error before producing the final submission.

There is no date or time variable in the supplied data. The training and test sets contain the same stores, while exact product-store pairs in the test set are new. Given that structure, I used **5-fold shuffled KFold cross-validation** on the training data with `random_state=42`.

The same folds were used for every model comparison. The Kaggle public score was not used to choose the model.

In [ ]:
cv = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(f"Using {N_FOLDS}-fold shuffled cross-validation with random_state={RANDOM_STATE}.")

## 7. RMSE calculation

RMSE is the competition metric. It measures the size of prediction errors and gives more weight to larger errors because the individual errors are squared before averaging.

I used the same RMSE calculation for every validation experiment.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

## 8. Mean-sales baseline

I started with a deliberately simple benchmark. For each validation fold, the model predicts the mean `total_sales` of the corresponding training portion for every validation observation.

This gives a reference point for judging whether the machine-learning models are actually learning useful structure from the predictors.

In [ ]:
y = train[TARGET].copy()

baseline_scores = []

for train_idx, valid_idx in cv.split(train):
    y_tr = y.iloc[train_idx]
    y_va = y.iloc[valid_idx]

    prediction = np.repeat(y_tr.mean(), len(valid_idx))
    baseline_scores.append(rmse(y_va, prediction))

baseline_mean = np.mean(baseline_scores)
baseline_std = np.std(baseline_scores, ddof=0)

print("Baseline fold RMSEs:", np.round(baseline_scores, 2))
print(f"Baseline mean CV RMSE: {baseline_mean:.4f} ± {baseline_std:.4f}")

## 9. Ridge regression benchmark

I then fitted Ridge regression as a conventional linear benchmark.

Numeric variables were median-imputed and standardised. Categorical variables were imputed and one-hot encoded. The `id` column was excluded because it is an identifier rather than a meaningful predictor.

This benchmark provides a useful comparison with a nonlinear categorical model.

In [ ]:
X = train.drop(columns=[TARGET, ID_COL]).copy()

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

ridge_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_cols
        )
    ]
)

ridge_pipeline = Pipeline([
    ("preprocessor", ridge_preprocessor),
    ("model", Ridge(alpha=10.0))
])

ridge_scores = np.sqrt(
    -cross_val_score(
        ridge_pipeline,
        X,
        y,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )
)

ridge_mean = ridge_scores.mean()
ridge_std = ridge_scores.std(ddof=0)

print("Ridge fold RMSEs:", np.round(ridge_scores, 2))
print(f"Ridge mean CV RMSE: {ridge_mean:.4f} ± {ridge_std:.4f}")

## 10. CatBoost models

The data contain several categorical variables, including `product_code` and `store_code`, as well as nonlinear relationships between the predictors and sales.

I therefore tested CatBoost because it can work directly with categorical variables and capture nonlinear effects without requiring one-hot encoding for every category.

I first evaluated a base configuration, then made a modest tuning change to the number of iterations, tree depth, learning rate and L2 regularisation. Both configurations were evaluated using exactly the same five validation folds.

In [ ]:
X_cb = train.drop(columns=[TARGET, ID_COL]).copy()

cat_cols = X_cb.select_dtypes(include=["object", "category"]).columns.tolist()

for col in cat_cols:
    X_cb[col] = X_cb[col].fillna("Missing").astype(str)

catboost_configs = {
    "CatBoost_Base": {
        "iterations": 500,
        "depth": 7,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    "CatBoost_Tuned": {
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.03,
        "l2_leaf_reg": 5
    }
}

catboost_results = {}
catboost_fold_scores = {}

for name, params in catboost_configs.items():
    scores = []
    start = time.time()

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_cb), start=1):
        model = CatBoostRegressor(
            **params,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1
        )

        model.fit(
            X_cb.iloc[train_idx],
            y.iloc[train_idx],
            cat_features=cat_cols
        )

        pred = model.predict(X_cb.iloc[valid_idx])
        scores.append(rmse(y.iloc[valid_idx], pred))

    catboost_fold_scores[name] = scores
    catboost_results[name] = {
        "CV_RMSE_Mean": np.mean(scores),
        "CV_RMSE_Std": np.std(scores, ddof=0),
        "Runtime_seconds": time.time() - start
    }

catboost_results_df = (
    pd.DataFrame(catboost_results)
    .T
    .sort_values("CV_RMSE_Mean")
)

display(catboost_results_df.round(4))

## 11. Model comparison and selection

The models are compared using mean cross-validation RMSE. I selected the model with the lowest mean CV RMSE among the models tested.

This decision was made before looking at the Kaggle public score, so the external test result did not influence model selection.

In [ ]:
comparison_rows = [
    ["Mean baseline", baseline_mean, baseline_std],
    ["Ridge", ridge_mean, ridge_std],
]

for name in catboost_results_df.index:
    comparison_rows.append([
        name,
        catboost_results[name]["CV_RMSE_Mean"],
        catboost_results[name]["CV_RMSE_Std"]
    ])

model_comparison = pd.DataFrame(
    comparison_rows,
    columns=["Model", "CV_RMSE_Mean", "CV_RMSE_Std"]
).sort_values("CV_RMSE_Mean")

display(model_comparison.round(4))

selected_model_name = model_comparison.iloc[0]["Model"]
print("Selected model:", selected_model_name)

model_comparison.to_csv(
    OUTPUT_DIR / "cv_model_comparison.csv",
    index=False
)

### Experiment record

The experiment log records the change made at each stage, why it was tested, the validation result and the resulting decision. This keeps the modelling process traceable rather than treating the final model as an unexplained choice.

In [ ]:
experiment_log = pd.DataFrame([
    {
        "Experiment": "E00",
        "Change": "Mean prediction baseline",
        "Reason": "Establish a simple reference point",
        "Validation": "5-fold KFold RMSE",
        "Result": baseline_mean,
        "Decision": "Reference only"
    },
    {
        "Experiment": "E01",
        "Change": "Ridge regression with one-hot categorical variables",
        "Reason": "Provide a conventional linear benchmark",
        "Validation": "5-fold KFold RMSE",
        "Result": ridge_mean,
        "Decision": "Compare with nonlinear model"
    },
    {
        "Experiment": "E02",
        "Change": "CatBoost base configuration",
        "Reason": "Handle categorical variables and nonlinear effects",
        "Validation": "5-fold KFold RMSE",
        "Result": catboost_results["CatBoost_Base"]["CV_RMSE_Mean"],
        "Decision": "Retain as candidate"
    },
    {
        "Experiment": "E03",
        "Change": "CatBoost modest tuning",
        "Reason": "Test a learning-rate, depth and regularisation trade-off",
        "Validation": "5-fold KFold RMSE",
        "Result": catboost_results["CatBoost_Tuned"]["CV_RMSE_Mean"],
        "Decision": "Selected because it had the lowest CV RMSE"
    }
])

display(experiment_log.round(4))
experiment_log.to_csv(OUTPUT_DIR / "experiment_log.csv", index=False)

## 12. Final model training

The tuned CatBoost configuration produced the lowest mean CV RMSE among the tested models, so I used it for the final training run.

For the final fit, the model was trained on the complete training dataset rather than on individual validation folds. The same preprocessing treatment for categorical missing values was retained. The test set was then transformed in the same way and used only to generate predictions.

In [ ]:
final_params = catboost_configs[selected_model_name]

X_full = train.drop(columns=[TARGET, ID_COL]).copy()
X_test = test.drop(columns=[ID_COL]).copy()

cat_cols_final = X_full.select_dtypes(include=["object", "category"]).columns.tolist()

for col in cat_cols_final:
    X_full[col] = X_full[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

final_model = CatBoostRegressor(
    **final_params,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1
)

start = time.time()
final_model.fit(
    X_full,
    y,
    cat_features=cat_cols_final
)
final_training_time = time.time() - start

test_predictions = final_model.predict(X_test)

submission = sample_submission.copy()
submission[TARGET] = test_predictions

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Final model:", selected_model_name)
print(f"Full-data training time: {final_training_time:.2f} seconds")
print("Submission saved to:", submission_path.resolve())

## 13. Submission checks

Before submitting to Kaggle, I verified that the prediction file has the expected number of rows and columns and that the predictions contain no missing or infinite values.

These checks are separate from model performance. They confirm that the generated file is structurally suitable for submission.

In [ ]:
print("Submission shape:", submission.shape)
print("Expected rows:", len(test))
print("Columns:", submission.columns.tolist())
print("Missing predictions:", submission[TARGET].isna().sum())
print("Infinite predictions:", np.isinf(submission[TARGET]).sum())

print("\nPrediction summary:")
display(submission[TARGET].describe())

print("\nFirst 10 predictions:")
display(submission.head(10))

assert submission.shape[0] == len(test)
assert submission.columns.tolist() == ["id", TARGET]
assert submission[TARGET].isna().sum() == 0
assert np.isinf(submission[TARGET]).sum() == 0
assert submission["id"].equals(sample_submission["id"])

print("\nAll submission checks passed.")

## 14. Kaggle evaluation and reproducibility record

The generated `outputs/submission.csv` was submitted to the DSN Kaggle competition after the checks above.

The submission returned a **public leaderboard RMSE of 1079.58071**.

This is an external evaluation and is therefore kept separate from the internal 5-fold CV RMSE of **1078.3807**. The Kaggle score was not used to select the model.

The final project outputs saved by this notebook are:

- `outputs/submission.csv` for the competition submission
- `outputs/cv_model_comparison.csv` for the model comparison
- `outputs/experiment_log.csv` for the modelling record

The final model was selected from the tested configurations rather than from a broad hyperparameter search. The results therefore describe the performance of the models evaluated in this notebook, not an exhaustive search of all possible approaches.

## Conclusion

The modelling process moved from a simple mean-sales benchmark to a linear Ridge model and then to CatBoost models designed to handle the categorical and nonlinear structure of the data.

The mean baseline produced a CV RMSE of approximately **1697.72**, Ridge reduced this to approximately **1137.67**, and the tuned CatBoost configuration produced the lowest tested CV RMSE at **1078.38**.

I therefore used the tuned CatBoost model to train on the full training data and generate the final test predictions. The resulting Kaggle submission achieved a public RMSE of **1079.58071**.

The main limitation is that the internal validation used shuffled KFold because the supplied data contain no time variable. In addition, the model search was deliberately modest because the qualification submission had a fixed deadline. Further work could investigate alternative validation schemes and additional feature engineering, but those experiments were not part of the submitted model selection process.